# Certamen de laboratorio · Minería de Datos · Unidad II

**martes 27 de octubre de 2026** · **individual** · 120 minutos · 100 puntos · 40 % de la nota de práctica

**Nombre:** _escriba aquí su nombre_ · **RUT:** _escriba aquí su RUT_

## Antes de empezar (lea esto primero)

1. **Archivo → Guardar una copia en Drive.** Trabaje siempre en su copia.
2. Ejecute la **celda ▶ entregada** (la primera de código). Debe ver: *Datos cargados: 200 fichas (160 de entrenamiento y 40 de prueba), 300 pedidos, 40 semanas y 60 estudiantes.*
3. Cada tarea tiene siempre las mismas partes, en este orden:

| Ícono | Qué es | Qué hace usted |
|:-:|---|---|
| 📖 | **Contexto** | Entender el problema del negocio |
| 💡 | **Concepto clave** | Repasar la idea que hay detrás |
| 🎯 | **Qué hacer** | Seguir los pasos |
| 📦 🔍 | **Qué va a obtener y qué representa** | Saber de antemano qué resultado buscar y qué significa para el caso |
| 🧩 | **Función con su firma** | Completar **solo el cuerpo**. No cambie el nombre ni los parámetros. El docstring dice qué recibe, qué devuelve y trae un ejemplo |
| 🧪 | **Prueba** con un ejemplo mínimo | Ejecutarla. Debe dar **exactamente** el resultado indicado, que se explica al lado. Si no da, corrija la función antes de seguir |
| 📊 | **Análisis** con los datos del laboratorio | Escribir el código que pide la tarea |
| ✅ | **Resultado esperado** | Comparar: dice **qué debe mostrar** su celda 📊 y en qué formato |
| ✍️ | **Respuesta** | Responder con **sus propios números** |

4. **De los datos al conocimiento.** Cada tarea termina con una pregunta 🧠. El curso trabaja con una escalera de tres peldaños:

| Peldaño | Qué es | Ejemplo en la clínica |
|---|---|---|
| **Dato** | Un hecho suelto | «La ficha 1023 recibió correo, vive lejos y no asistió» |
| **Información** | Datos procesados que muestran un patrón | «El árbol acierta 9 de cada 10 fichas de prueba» |
| **Conocimiento** | Información convertida en una **decisión** | «Hay que llamar a quienes reciben correo y viven lejos» |

   Su código produce **información**; sus respuestas 🧠 deben llegar al **conocimiento**.

5. Puede usar **todos los recursos** (apuntes, guías, internet, IA). Pero se evalúa que **interprete sus resultados**: una respuesta genérica, sin sus números, no tiene puntaje.
6. Tiempo sugerido: **Parte I** hasta el minuto 40, **Parte II** hasta el 80 y **Parte III** hasta el 120.
7. Al terminar: **Archivo → Descargar → Descargar .ipynb**, y súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.

| Parte | Caso | Tareas | Puntos |
|---|---|:-:|:-:|
| I | Clínica veterinaria: ZeroR, OneR y árbol | 1 a 3 | 35 |
| II | Ferretería online: soporte, confianza, lift y regresión | 4 a 6 | 35 |
| III | Liceo: k-means por rondas y sus gráficas | 7 y 8 | 30 |

In [ ]:
# ▶ CELDA ENTREGADA — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/certamen-lab/"
clinica = pd.read_csv(RUTA + "lab_clinica.csv")              # Parte I
pedidos = pd.read_csv(RUTA + "lab_pedidos.csv")              # Parte II (Apriori)
semanas = pd.read_csv(RUTA + "lab_semanas.csv")              # Parte II (regresión)
liceo = pd.read_csv(RUTA + "lab_liceo.csv")                  # Parte III

entrenamiento = clinica[clinica["Conjunto"] == "entrenamiento"]
prueba = clinica[clinica["Conjunto"] == "prueba"]
canastas = pedidos.groupby("Pedido")["Producto"].apply(set).tolist()          # un conjunto por pedido
puntos = list(zip(liceo["Estudio"], liceo["Actividad"]))                     # pares (x, y)

ATRIBUTOS = ["Especie", "Recordatorio", "Distancia", "Edad"]
PRODUCTOS = ['Pintura', 'Brocha', 'Rodillo', 'Lija', 'Guantes', 'Huincha']
ESTRELLAS_INICIO = [(1, 9), (2, 1), (3, 3)]

print(f"Datos cargados: {len(clinica)} fichas ({len(entrenamiento)} de entrenamiento y {len(prueba)} de prueba), "
      f"{len(canastas)} pedidos, {len(semanas)} semanas y {len(puntos)} estudiantes.")

---
# Parte I · Clínica veterinaria (35 puntos)

La clínica quiere predecir **qué dueños no traerán a su mascota al control de vacunas**. La tabla `clinica` tiene **200 fichas** con:

| Columna | Valores |
|---|---|
| `Ficha` | número de ficha (1001 a 1200) |
| `Especie` | Perro · Gato |
| `Recordatorio` | WhatsApp · Correo · Ninguno |
| `Distancia` | Cerca · Lejos |
| `Edad` | Cachorro · Adulto · Senior |
| `Asistio` | **Sí · No** ← la clase que se quiere predecir |
| `Conjunto` | `entrenamiento` (160 fichas, para **construir** los modelos) · `prueba` (40 fichas, para **evaluarlos**) |

Ya están separadas en las variables **`entrenamiento`** y **`prueba`**. Los modelos se construyen **solo** con `entrenamiento`.

## Tarea 1 · ZeroR: la línea base (8 puntos)

**📖 Contexto.** Antes de construir un modelo, la clínica necesita saber **cuánto se acierta «sin pensar»**: un modelo solo vale la pena si **mejora** eso.

> **💡 Concepto clave.** **ZeroR** no mira ningún atributo: predice siempre la **clase mayoritaria**. Su acierto es la **línea base**, el mínimo que cualquier modelo debe superar. **Acierto** = aciertos / total.

### 🎯 Qué hacer

1. Complete `zeror(df, clase)` y ejecute la prueba 🧪.
2. En 📊: aplíquela a `entrenamiento` y calcule su **acierto en `prueba`**: ZeroR predice la clase mayoritaria para las 40 fichas de prueba. El acierto se calcula así: `(prueba["Asistio"] == clase_mayoritaria).mean()`.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| La clase que predice ZeroR, sus **errores en entrenamiento** (de 160) y su **acierto en prueba** (entre 0 y 1). | El acierto en prueba es **la vara con que se medirán los demás modelos**: lo que la clínica lograría sin analizar ningún dato. |

In [ ]:
# 🧩 Tarea 1 · complete el cuerpo de la(s) función(es)
def zeror(df, clase):
    """ZeroR: predice SIEMPRE la clase que más se repite.

    Recibe:   df    → la tabla (DataFrame)
              clase → el nombre de la columna que se quiere predecir, aquí "Asistio"
    Devuelve: (clase_mayoritaria, errores)
              errores = cuántas filas NO son de la clase mayoritaria (ahí ZeroR se equivoca)
    Ejemplo:  si hay 5 filas «Sí» y 3 «No»  →  ("Sí", 3)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 1 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
c, e = zeror(mini, "Asistio")
print(c, e)
# ✅ Debe mostrar: Sí 3

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Sí 3`**: en las 8 fichas del ejemplo hay 5 «Sí» y 3 «No»: ZeroR predice «Sí» y falla en las 3 «No». Si le da 5, está contando aciertos.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 1 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar la clase que predice ZeroR, sus **errores en entrenamiento** (de 160) y su **acierto en prueba** (entre 0 y 1).

### ✍️ Responda

**1. Sus resultados:** ¿Qué predice ZeroR y qué acierto obtiene en prueba?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si la clínica supusiera que todos vienen, ¿en cuántas de cada 10 mascotas se equivocaría?

_Su respuesta:_

## Tarea 2 · OneR: ¿qué dato explica más la asistencia? (12 puntos)

**📖 Contexto.** La administradora quiere saber **qué dato de la ficha** (especie, recordatorio, distancia o edad) explica mejor si el dueño viene.

> **💡 Concepto clave.** **OneR** prueba cada atributo por separado: para cada valor, predice la clase más frecuente en ese valor; los **errores** son las fichas de la otra clase. Gana el atributo con **menos errores**. Un atributo cuya regla dice lo mismo en todos sus valores **no aporta**: equivale a ZeroR.

### 🎯 Qué hacer

1. Complete `oner(df, atributo, clase)` y ejecute la prueba 🧪.
2. En 📊: aplique `oner()` a `entrenamiento` con **cada uno** de los 4 `ATRIBUTOS`, muestre una tabla con sus errores, identifique el **ganador** y muestre **su regla**.
3. Calcule el **acierto en prueba** de la regla ganadora: para cada ficha de prueba, la predicción es `reglas[ficha[ganador]]`.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Una **tabla de 4 filas** (atributo y errores), el **atributo ganador**, **su regla** (un valor → una clase) y su **acierto en prueba**. | El ganador es **el dato que más ayuda a predecir la asistencia**. Su regla ya es una política simple que la clínica podría aplicar. |

In [ ]:
# 🧩 Tarea 2 · complete el cuerpo de la(s) función(es)
def oner(df, atributo, clase):
    """OneR con UN atributo: una regla por cada valor del atributo.

    Para cada valor del atributo: la regla es la clase que más se repite en ese valor, y los
    errores son las filas de ese valor que tienen la otra clase. Se suman los errores.
    Recibe:   df, atributo (p. ej. "Recordatorio"), clase ("Asistio")
    Devuelve: (reglas, errores)
              reglas = diccionario {valor: clase predicha}
    Ejemplo:  si con "Correo" hay 2 «Sí» y 1 «No», su regla es "Correo" → "Sí" y suma 1 error.
              Devuelve algo como ({"Correo": "Sí", "Ninguno": "No", "WhatsApp": "Sí"}, 1)
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 2 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
reglas, e = oner(mini, "Recordatorio", "Asistio")
print(dict(sorted(reglas.items())), e)
# ✅ Debe mostrar: {'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 1

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`{'Correo': 'Sí', 'Ninguno': 'No', 'WhatsApp': 'Sí'} 1`**: Correo: 2 Sí y 1 No (regla Sí, 1 error); Ninguno: 0 Sí y 2 No (regla No, 0 errores); WhatsApp: 3 Sí y 0 No (regla Sí, 0 errores). Total: 1 error.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 2 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar una **tabla de 4 filas** (atributo y errores), el **atributo ganador**, **su regla** (un valor → una clase) y su **acierto en prueba**.

### ✍️ Responda

**1. Sus resultados:** ¿Qué atributo gana, cuál es su regla y cuánto acierta en prueba comparado con ZeroR? ¿Hay algún atributo que no le gane a ZeroR? ¿Por qué?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Con la regla ganadora, ¿qué le recomendaría a la clínica sobre **cómo enviar los recordatorios**?

_Su respuesta:_

## Tarea 3 · Árbol: una segunda pregunta donde hace falta (15 puntos)

**📖 Contexto.** La regla de OneR se equivoca casi siempre en **una misma rama**. Un **árbol** agrega una **segunda pregunta solo en esa rama**, para separar mejor.

> **💡 Concepto clave.** Un **árbol de decisión** encadena preguntas. Una **rama pura** tiene una sola clase; una **rama impura** mezcla clases. En la rama impura se busca la segunda pregunta **usando solo las fichas de esa rama**: es aplicar OneR de nuevo, pero a un grupo más chico.

### 🎯 Qué hacer

1. En 📊: para **cada valor** del ganador, tome las fichas de entrenamiento con ese valor y cuente sus errores con `zeror()`. La rama con **más errores** es la impura.
2. Con **solo las fichas de esa rama**, aplique `oner()` a los otros atributos y elija el de **menos errores**: es la segunda pregunta.
3. Calcule el **acierto en prueba** del árbol con la función entregada `predecir_arbol()` y compárelo con ZeroR y OneR.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Los **errores de cada rama**, la **rama impura**, la **segunda pregunta** con su regla, y una **tabla de acierto en prueba** de ZeroR, OneR y el árbol. | El árbol es un **protocolo de dos preguntas**. Si supera a OneR en prueba, combinar dos datos predice mejor que uno solo. |

In [ ]:
# ▶ FUNCIÓN ENTREGADA — no la modifique. Clasifica una ficha con el árbol de dos preguntas.
def predecir_arbol(fila, raiz, reglas_raiz, rama, atributo2, reglas2):
    """Si la ficha cae en la rama impura, decide la 2.ª pregunta; si no, decide la regla de la raíz.

    Uso: predecir_arbol(fila, ganador, reglas_ganador, rama, atributo2, reglas2)
         con ganador y reglas_ganador de la tarea 2, y rama, atributo2 y reglas2 de esta tarea.
    """
    if fila[raiz] == rama:
        return reglas2.get(fila[atributo2], reglas_raiz[rama])
    return reglas_raiz[fila[raiz]]

In [ ]:
# 🧪 Prueba de la tarea 3 con un ejemplo mínimo: ejecútela sin cambiarla
mini = pd.DataFrame({
    "Recordatorio": ["WhatsApp", "WhatsApp", "WhatsApp", "Correo", "Correo", "Correo", "Ninguno", "Ninguno"],
    "Distancia":    ["Cerca",    "Lejos",    "Cerca",    "Cerca",  "Cerca",  "Lejos",  "Cerca",   "Lejos"],
    "Asistio":      ["Sí",       "Sí",       "Sí",       "Sí",     "Sí",     "No",     "No",      "No"]})
rama_mini = mini[mini["Recordatorio"] == "Correo"]
c, e = zeror(rama_mini, "Asistio")
r, e2 = oner(rama_mini, "Distancia", "Asistio")
print(c, e, dict(sorted(r.items())), e2)
# ✅ Debe mostrar: Sí 1 {'Cerca': 'Sí', 'Lejos': 'No'} 0

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`Sí 1 {'Cerca': 'Sí', 'Lejos': 'No'} 0`**: en el ejemplo, la rama Correo es la impura (2 Sí y 1 No → 1 error; las otras ramas tienen 0). Dentro de ella, Distancia separa sin errores: las 2 fichas Cerca asistieron y la Lejos no.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 3 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar los **errores de cada rama**, la **rama impura**, la **segunda pregunta** con su regla, y una **tabla de acierto en prueba** de ZeroR, OneR y el árbol.

### ✍️ Responda

**1. Sus resultados:** ¿Cuál es la rama impura y cuál la segunda pregunta? ¿Qué modelo acierta más en prueba? ¿Por qué se mide en prueba y no en entrenamiento?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Traduzca el árbol a un **protocolo para la recepcionista**: ¿a qué dueños hay que llamar antes del control?

_Su respuesta:_

---
# Parte II · Ferretería online (35 puntos)

**Apriori.** La tabla `pedidos` tiene una fila por producto comprado (`Pedido`, `Producto`). Ya está convertida en **`canastas`**: una lista de **300 conjuntos**, uno por pedido. Por ejemplo, `{'Pintura', 'Brocha', 'Guantes'}`. Los 6 productos están en `PRODUCTOS`.

📄 **Antes de la tarea 4, lea la guía de apoyo «soporte(), confianza() y lift()».**

**Regresión.** La tabla `semanas` tiene 40 semanas con `Avisos` (gasto en avisos, en miles de $), `Cyber` (1 si hubo evento Cyber esa semana, 0 si no) y `Pedidos` (pedidos de esa semana).

## Tarea 4 · Soporte: ¿qué se vende lo suficiente? (8 puntos)

**📖 Contexto.** La ferretería quiere la sección **«Otros clientes también llevaron…»**. Primero necesita saber qué productos aparecen lo suficiente como para analizarlos.

> **💡 Concepto clave.** El **soporte** es la proporción de pedidos que contienen un producto (o varios juntos). Con un **soporte mínimo** se descartan los productos raros: si un producto casi no se vende, ninguna regla con él será frecuente.

### 🎯 Qué hacer

1. Lea la **guía de apoyo** «soporte(), confianza() y lift()».
2. Complete `soporte(canastas, productos)` y ejecute la prueba 🧪.
3. En 📊: calcule el soporte de los **6 productos** y marque cuáles llegan al **10 %**.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Una **tabla de 6 filas**: producto, soporte y si es frecuente (≥ 10 %). | Muestra **qué tan populares** son los productos. Los no frecuentes quedan fuera del análisis, y un producto muy popular puede **engañar** a la confianza (lo verá en la tarea 5). |

In [ ]:
# 🧩 Tarea 4 · complete el cuerpo de la(s) función(es)
def soporte(canastas, productos):
    """Soporte: en qué proporción de TODOS los pedidos aparecen juntos los productos dados.

    Recibe:   canastas  → lista de pedidos; cada pedido es un conjunto, p. ej. {"Pintura", "Guantes"}
              productos → lista con uno o más productos, p. ej. ["Pintura"] o ["Pintura", "Brocha"]
    Devuelve: un número entre 0 y 1
    Ejemplo:  con 4 pedidos, si "Pan" está en 3   →  soporte(canastas, ["Pan"]) = 0.75
              si "Pan" y "Queso" están juntos en 2  →  soporte(canastas, ["Pan", "Queso"]) = 0.5
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 4 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Pan", "Queso"}, {"Pan"}, {"Pan", "Queso", "Vino"}, {"Vino"}]
print(soporte(canastas_mini, ["Pan"]), soporte(canastas_mini, ["Pan", "Queso"]))
# ✅ Debe mostrar: 0.75 0.5

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.75 0.5`**: Pan está en 3 de los 4 pedidos del ejemplo; Pan y Queso juntos, en 2 de 4.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 4 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar una **tabla de 6 filas**: producto, soporte y si es frecuente (≥ 10 %).

### ✍️ Responda

**1. Sus resultados:** ¿Qué producto no es frecuente? ¿Qué producto aparece en casi todos los pedidos?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** ¿Vale la pena armar promociones con el producto no frecuente? ¿Por qué?

_Su respuesta:_

## Tarea 5 · Confianza y lift: ¿relación real o casualidad? (12 puntos)

**📖 Contexto.** Una regla **A → B** dice «quien lleva A también lleva B». La ferretería quiere saber **qué regla usar para recomendar**.

> **💡 Concepto clave.** **Confianza(A → B)**: de los pedidos con A, qué proporción tiene B. **Lift** = confianza / soporte de B: compara con la popularidad de B. **Lift > 1**: relación real. **Lift ≈ 1 o menos**: no hay relación, aunque la confianza sea alta.

### 🎯 Qué hacer

1. Complete `confianza(canastas, a, b)` y `lift(canastas, a, b)` y ejecute la prueba 🧪.
2. En 📊: calcule **soporte, confianza y lift** de **Pintura → Brocha**, **Brocha → Pintura** y **Rodillo → Guantes**.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Una **tabla de 3 reglas** con soporte, confianza y lift. | La **confianza** dice qué tan seguido se cumple la regla; el **lift** dice si la regla es **útil para recomendar**. |

In [ ]:
# 🧩 Tarea 5 · complete el cuerpo de la(s) función(es)
def confianza(canastas, a, b):
    """Confianza de la regla a → b: de los pedidos que tienen a, qué proporción tiene también b.

    Confianza(a → b) = soporte(a y b) / soporte(a)      (se divide por el producto de la IZQUIERDA)
    Ejemplo:  si 3 pedidos tienen Pan y 2 de ellos también Queso  →  confianza(Pan → Queso) = 2/3
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def lift(canastas, a, b):
    """Lift de la regla a → b: cuántas veces más se lleva b cuando se lleva a, comparado con el azar.

    Lift(a → b) = confianza(a → b) / soporte(b)
    Lectura:  mayor que 1 → asociación real · igual a 1 → no hay relación · menor que 1 → se evitan
    Ejemplo:  confianza(Pan → Queso) = 0.667 y Queso está en la mitad de los pedidos (0.5)
              →  lift = 0.667 / 0.5 = 1.33
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 5 con un ejemplo mínimo: ejecútela sin cambiarla
canastas_mini = [{"Pan", "Queso"}, {"Pan"}, {"Pan", "Queso", "Vino"}, {"Vino"}]
print(round(confianza(canastas_mini, "Pan", "Queso"), 4), round(lift(canastas_mini, "Pan", "Queso"), 4), round(lift(canastas_mini, "Vino", "Pan"), 4))
# ✅ Debe mostrar: 0.6667 1.3333 0.6667

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`0.6667 1.3333 0.6667`**: de 3 pedidos con Pan, 2 llevan Queso (0,67); Queso está en la mitad de los pedidos → lift 0,67 / 0,5 = 1,33 (asociación). Vino → Pan: de 2 pedidos con Vino, 1 lleva Pan (0,5) y Pan está en 3 de 4 → lift 0,67 (menor que 1: no hay asociación).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 5 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar una **tabla de 3 reglas** con soporte, confianza y lift.

### ✍️ Responda

**1. Sus resultados:** ¿Qué regla tiene **más confianza**? ¿Y **más lift**? ¿Por qué no coinciden?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Si un cliente pone **Rodillo** en el carro, ¿conviene mostrarle Guantes? ¿Qué regla usaría para recomendar?

_Su respuesta:_

## Tarea 6 · Regresión: ¿cuántos pedidos tendremos? (15 puntos)

**📖 Contexto.** La ferretería quiere **planificar stock y despachos** según cuánto invierta en avisos y si hay evento Cyber.

> **💡 Concepto clave.** La **regresión lineal** ajusta y = a + b₁·x₁ + b₂·x₂. Cada **coeficiente** dice cuánto cambia y cuando esa variable sube en 1. Cyber vale 0 o 1: su coeficiente se suma solo en semanas con Cyber. El modelo solo vale **dentro del rango** de datos con que se ajustó; usarlo fuera es **extrapolar**.

### 🎯 Qué hacer

1. En 📊: con `X = semanas[["Avisos", "Cyber"]]` e `y = semanas["Pedidos"]`, ajuste un `LinearRegression()` y muestre `modelo.intercept_`, `modelo.coef_` y el R² (`modelo.score(X, y)`).
2. Prediga los pedidos con **$60 mil** en avisos **sin** y **con** Cyber: `modelo.predict(pd.DataFrame({"Avisos": [60, 60], "Cyber": [0, 1]}))`.
3. Muestre el **mínimo y el máximo** de `Avisos`.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| La **ecuación** (intercepto y dos coeficientes), el **R²**, **dos predicciones** y el **rango** de Avisos observado. | Los coeficientes son **efectos en pedidos reales**: cuánto rinde cada $1.000 en avisos y cuánto suma un Cyber. El rango dice **hasta dónde se puede confiar** en el modelo. |

In [ ]:
# 📊 Tarea 6 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar la **ecuación** (intercepto y dos coeficientes), el **R²**, **dos predicciones** y el **rango** de Avisos observado.

### ✍️ Responda

**1. Sus resultados:** Explique qué significa cada coeficiente para el dueño. ¿Cuántos pedidos se esperan con $60 mil, sin y con Cyber?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** El dueño quiere gastar **$500 mil** en avisos y usar el modelo para saber cuántos pedidos tendrá. ¿Qué le diría?

_Su respuesta:_

---
# Parte III · Liceo (30 puntos)

La tabla `liceo` tiene **60 estudiantes** con `Estudio` (horas de estudio a la semana) y `Actividad` (horas de actividad física a la semana). Ya están en **`puntos`**: una lista de pares `(x, y)`.

Se buscan **3 grupos** con k-means, partiendo de las estrellas **`ESTRELLAS_INICIO` = (1; 9), (2; 1) y (3; 3)**.

**Recuerde:** una **ronda** = **(1) elegir:** cada estudiante va a la estrella más cercana + **(2) moverse:** cada estrella va al promedio de su grupo. Se repite hasta que en una ronda **nadie cambie de grupo**.

## Tarea 7 · Una ronda de k-means: elegir y moverse (12 puntos)

**📖 Contexto.** La orientadora quiere formar **3 grupos de estudiantes parecidos** para ofrecer talleres distintos, sin categorías definidas de antemano.

> **💡 Concepto clave.** **k-means** agrupa sin etiquetas. Cada grupo tiene una **estrella** (su centro). Una **ronda** tiene dos pasos: **elegir** (cada estudiante va a la estrella más cercana; se compara d² = (x − x★)² + (y − y★)², sin raíz) y **moverse** (cada estrella va al promedio de su grupo).

### 🎯 Qué hacer

1. Complete `asignar(puntos, estrellas)` (paso **elegir**) y `actualizar(puntos, grupos, estrellas)` (paso **moverse**).
2. Ejecute la prueba 🧪.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| Que la prueba 🧪 dé **exactamente** lo indicado: los grupos de la ronda 1 y las nuevas estrellas. | Estas dos funciones **son una ronda completa**: en la tarea 8 se repiten hasta que los grupos se estabilizan. |

In [ ]:
# 🧩 Tarea 7 · complete el cuerpo de la(s) función(es)
def asignar(puntos, estrellas):
    """Paso 1 de una ronda (ELEGIR): cada punto elige la estrella más cercana.

    Recibe:   puntos    → lista de pares (x, y), p. ej. [(4, 9), (7, 4), ...]
              estrellas → lista de pares (x, y), una por grupo
    Devuelve: lista de grupos, uno por punto: 0 = primera estrella, 1 = segunda, ...
    Para comparar basta d² = (x − x★)² + (y − y★)², sin raíz.
    Ejemplo:  asignar([(1, 1), (8, 9)], [(3, 3), (10, 10)])  →  [0, 1]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def actualizar(puntos, grupos, estrellas):
    """Paso 2 de una ronda (MOVERSE): cada estrella se va al promedio de los puntos de su grupo.

    Recibe:   puntos, grupos (lo que devuelve asignar) y las estrellas actuales
    Devuelve: lista de nuevas estrellas (x, y). Si un grupo quedara vacío, su estrella no se mueve.
    Ejemplo:  actualizar([(1, 1), (3, 1), (8, 9)], [0, 0, 1], [(0, 0), (10, 10)])  →  [(2.0, 1.0), (8.0, 9.0)]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 7 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
print(asignar(puntos_mini, [(3, 3), (10, 10)]))
# ✅ Debe mostrar: [0, 0, 0, 1, 1]
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
nuevas = actualizar(puntos_mini, [0, 0, 0, 1, 1], [(3, 3), (10, 10)])
print([(round(float(x), 2), round(float(y), 2)) for x, y in nuevas])
# ✅ Debe mostrar: [(3.0, 2.67), (8.5, 8.5)]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`[0, 0, 0, 1, 1]`**: con ★1 en (3; 3) y ★2 en (10; 10), los tres primeros puntos quedan más cerca de ★1 (el (6; 6) por poco: d² 18 contra 32) y los dos últimos de ★2.
- **`[(3.0, 2.67), (8.5, 8.5)]`**: ★1 se mueve al promedio de (1; 1), (2; 1) y (6; 6) = (3; 2,67); ★2, al de (8; 9) y (9; 8) = (8,5; 8,5).

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

## Tarea 8 · El ciclo de rondas y sus gráficas (18 puntos)

**📖 Contexto.** Una ronda no basta: al moverse las estrellas, algunos estudiantes quedan más cerca de otra y **cambian de grupo**. Hay que repetir hasta que los grupos no cambien.

> **💡 Concepto clave.** El algoritmo termina cuando, en una ronda, **nadie cambia de grupo**. Las gráficas de cada ronda muestran cómo las estrellas **se acercan al centro de cada grupo**.

### 🎯 Qué hacer

1. Complete `graficar_ronda(...)` (puntos coloreados por grupo, estrella ★ grande del mismo color, ejes de 0 a 10 con nombre, título con la ronda) y `kmeans_por_rondas(...)` (repite asignar → contar cambios → graficar → actualizar, hasta 0 cambios).
2. Ejecute la prueba 🧪 (sin gráficas).
3. En 📊: córralo con los **60 estudiantes** desde `ESTRELLAS_INICIO` con `graficar=True`, y muestre **cuántos cambian en cada ronda**, las **estrellas finales** y **cuántos estudiantes** hay en cada grupo.

| 📦 Qué va a obtener | 🔍 Qué representa |
|---|---|
| **Una gráfica por ronda**, la lista de **cambios por ronda** (la última debe ser 0), las **3 estrellas finales** y el **tamaño de cada grupo** (suman 60). | Las estrellas finales son el **estudiante típico de cada grupo**: con ellas la orientadora puede nombrar los grupos y decidir talleres. |

In [ ]:
# 🧩 Tarea 8 · complete el cuerpo de la(s) función(es)
def graficar_ronda(puntos, grupos, estrellas, titulo):
    """Dibuja UNA ronda: los puntos coloreados según su grupo y cada estrella como ★ grande.

    Requisitos de la gráfica:
      • cada grupo con un color distinto, y su estrella del MISMO color, con marcador "*" grande
      • ejes de 0 a 10, con nombre: "Horas de estudio" (x) y "Horas de actividad física" (y)
      • el título que se recibe (debe decir el número de ronda)
    No devuelve nada: solo muestra la gráfica (plt.show()).
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass


def kmeans_por_rondas(puntos, estrellas_inicio, graficar=True):
    """El CICLO de k-means: repite rondas (elegir + moverse) hasta que en una ronda NADIE cambie.

    En cada ronda:
      1. grupos = asignar(puntos, estrellas)
      2. se cuenta cuántos puntos cambiaron de grupo respecto de la ronda anterior
         (en la ronda 1 no hay con qué comparar: cambios = None)
      3. se guarda la ronda en el historial y, si graficar=True, se grafica con graficar_ronda()
      4. si cambios == 0 → se termina; si no → estrellas = actualizar(...) y se sigue
    Devuelve: (estrellas_finales, grupos_finales, historial)
              historial = lista con un diccionario por ronda: {"ronda": 1, "cambios": None, "estrellas": [...]}
    Ejemplo:  si en la ronda 2 cambia un punto y en la 3 ninguno  →  cambios [None, 1, 0]
    """
    # ✍️ Escriba aquí el cuerpo de la función
    pass

In [ ]:
# 🧪 Prueba de la tarea 8 con un ejemplo mínimo: ejecútela sin cambiarla
puntos_mini = [(1, 1), (2, 1), (6, 6), (8, 9), (9, 8)]
estrellas_m, grupos_m, historial_m = kmeans_por_rondas(puntos_mini, [(3, 3), (10, 10)], graficar=False)
print(len(historial_m), [h["cambios"] for h in historial_m])
# ✅ Debe mostrar: 3 [None, 1, 0]

**✅ Resultado esperado de la prueba 🧪 y por qué:**

- **`3 [None, 1, 0]`**: 3 rondas: en la 1 no hay con qué comparar (None); en la 2 el punto (6; 6) se pasa a ★2, porque ★1 se alejó a (3; 2,67) y ★2 se acercó a (8,5; 8,5) (1 cambio); en la 3 nadie cambia (0) → termina.

*Si su resultado es distinto, la función tiene un error: corríjala antes de seguir.*

In [ ]:
# 📊 Tarea 8 · escriba aquí su análisis con los datos del laboratorio

**✅ Compare:** su celda 📊 debe mostrar **Una gráfica por ronda**, la lista de **cambios por ronda** (la última debe ser 0), las **3 estrellas finales** y el **tamaño de cada grupo** (suman 60).

### ✍️ Responda

**1. Sus resultados:** ¿En cuántas rondas termina y cuántos cambian en cada una? ¿Dónde quedan las estrellas y cuántos estudiantes tiene cada grupo?

_Su respuesta, con sus números:_

**2. 🧠 De la información al conocimiento:** Mirando las estrellas finales: ¿qué tipo de estudiante representa cada grupo? Póngales nombre y proponga **un taller** para uno.

_Su respuesta:_

---
## ✔️ Para entregar

1. **Entorno de ejecución → Reiniciar y ejecutar todo**: revise que todo corra sin errores.
2. **Archivo → Descargar → Descargar .ipynb**.
3. Súbalo al **Aula Virtual** como `Apellido_Nombre.ipynb`.